# Fase 4 v2 — Eventos intensos e extremos

Amostragem estratificada e estatísticas ponderadas para preservar a cauda da distribuição do radar.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/04_extremes')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## Estratos e pesos de amostragem

In [ ]:
strata = pd.read_parquet(OUT/'stratum_sampling.parquet')
display(strata)


## Comparação da prevalência ponderada com a referência exata da Fase 0

In [ ]:
prev = pd.read_parquet(OUT/'event_prevalence.parquet')
fixed = prev[prev.family.eq('fixed')].copy()
display(fixed[['event_id','threshold','weighted_event_rate_sampled_blocks','reference_event_rate_phase0','event_rate','event_rate_source','sample_event_count_unweighted','effective_sample_size_event']])


In [ ]:
plot = fixed.dropna(subset=['reference_event_rate_phase0'])
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(plot.threshold, plot.weighted_event_rate_sampled_blocks, marker='o', label='Fase 4 v2 ponderada')
ax.plot(plot.threshold, plot.reference_event_rate_phase0, marker='o', label='Fase 0 exata')
ax.set_yscale('log')
ax.set_xlabel('Limiar da legenda do radar')
ax.set_ylabel('Fração de pixels')
ax.legend()
ax.set_title('Cauda do radar: amostra ponderada × referência exata')
plt.tight_layout()
plt.show()


## Efeitos evento × não-evento

In [ ]:
effects = pd.read_parquet(OUT/'effect_sizes.parquet')
for event_id in ['fixed_ge_20','fixed_ge_30','fixed_ge_40','fixed_ge_45']:
    e = effects[effects.event_id.eq(event_id)].copy()
    e['abs_smd'] = e.standardized_mean_difference.abs()
    print('\n', event_id)
    display(e.sort_values('abs_smd', ascending=False).head(10)[[
        'predictor','source','event_mean','non_event_mean',
        'standardized_mean_difference','n_event_sample','effective_n_event'
    ]])


## Quantis ponderados dos valores positivos

In [ ]:
thr = pd.read_parquet(OUT/'threshold_definitions.parquet')
display(thr[thr.family.eq('positive_quantile')])


## Curvas de evento por decil do preditor

In [ ]:
dec = pd.read_parquet(OUT/'event_rate_by_predictor_decile.parquet')
event_id = 'fixed_ge_40'
for predictor in ['tcwv','r_500','t2m','t_850','delta_r_500_850']:
    d = dec[(dec.event_id.eq(event_id)) & (dec.predictor.eq(predictor))]
    if d.empty:
        continue
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(d.decile, d.event_rate, marker='o')
    ax.set_xlabel('Decil ponderado do preditor')
    ax.set_ylabel('P(evento) ponderada')
    ax.set_title(f'{event_id} × {predictor}')
    plt.tight_layout()
    plt.show()
